# Data Loader - Description

This notebook loads the data sources via API or stores data provided under /raw_data to .parquet files. The .parquet files generate by this notebook are referred to as "bronze" level.

## Data Sources

Taxi Data: https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data via API
Census Tract Data: https://data.cityofchicago.org/Facilities-Geographic-Boundaries/Census_Tracts/4hp8-2i8z/about_data 
Weather Data: https://mesonet.agron.iastate.edu/request/download.phtml?network=IL_ASOS
    https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?station=MDW&data=all&year1=2024&month1=1&day1=1&year2=2026&month2=5&day2=26&tz=America%2FChicago&format=onlycomma&latlon=yes&elev=yes&missing=null&trace=0.0001&direct=yes -> MDW

Chicago Landsmark Information (Points of interest): https://data.cityofchicago.org/Historic-Preservation/Individual-Landmarks-Map/bddq-yxar
Parks and Facilities: https://data.cityofchicago.org/Parks-Recreation/Parks-Facilities-Features-Shapefiles/thkh-m6bg/about_data - zip folder

Further possible
Bus and Rail Data: https://data.cityofchicago.org/Transportation/CTA-Ridership-Daily-Boarding-Totals/6iiy-9s97/about_data
Chicago Landsmark Information (Points of interest): https://data.cityofchicago.org/Historic-Preservation/Individual-Landmarks-Map/bddq-yxar


# Download Taxi Data via API
Load the data from "https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data" using SODA3 API and store it as CSV.

Pipeline:
SODA API → CSV as Raw Backup → DuckDB CSV → Parquet → Polars LazyFrame

Data Prep and Cleaning Pipeline:
raw CSV
  ↓
bronze Parquet      # 1:1 aus CSV/API, möglichst unverändert
  ↓
silver Parquet      # gecleant, typisiert, gefiltert
  ↓
gold/features       # ML-Features, train/test-ready

In [3]:
import pandas as pd
import requests
from pathlib import Path
import duckdb

import polars as pl
import h3
import geopandas as gpd

from shapely.geometry import Polygon, MultiPolygon

In [ ]:
SAMPLE_MODE = True

# API Parameter
DATASET_ID = "ajtu-isnz"
API_URL = f"https://data.cityofchicago.org/api/v3/views/ajtu-isnz/query.csv"
APP_TOKEN = "***"

output_path = Path("data/taxi_test_5k.csv")

# Query Parameter
LIMIT = 5_000

SELECT_FIELDS = []

WHERE_CLAUSE = ""

query = f"""
SELECT
    {", ".join(f"`{col}`" for col in SELECT_FIELDS)}
WHERE
    {WHERE_CLAUSE}
LIMIT {LIMIT}
"""

query = query if len(SELECT_FIELDS) > 0 and WHERE_CLAUSE != "" else f"""SELECT * LIMIT {LIMIT}"""

# Build payload
headers = {}

if APP_TOKEN and APP_TOKEN != "***":
    headers["X-App-Token"] = APP_TOKEN

payload = {
    "query": query
}

# CSV per Post streamen
with requests.post(
    API_URL,
    headers=headers,
    json=payload,
    stream=True,
    timeout=180
) as r:
    r.raise_for_status()

    with output_path.open("wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

print(f"Download finished: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

Download finished: taxi_test_5k.csv
File size: 2.61 MB


# Create Parquet File for efficient Data Handling

In [5]:
overwrite = False

raw_dir = Path("../data/raw_data")
processed_dir = Path("../data/processed_data")

if not raw_dir.exists():
    raise FileNotFoundError(f"Raw data folder does not exist: {raw_dir}")

processed_dir.mkdir(parents=True, exist_ok=True)

csv_files = sorted(raw_dir.glob("*.csv"))

if not csv_files:
    print(f"No CSV files found in {raw_dir}")

for csv_path in csv_files:
    parquet_path = processed_dir / f"bronze_{csv_path.stem}.parquet"

    if parquet_path.exists() and not overwrite:
        print(f"Skipping {csv_path.name}: {parquet_path.name} already exists")
        
        print("Quality check - Rows in parquet file:")
    
        duckdb.sql(f"""
        SELECT count(*)
        FROM read_parquet('{parquet_path}')
        """).show()
    
        continue

    print(f"Converting {csv_path.name} -> {parquet_path.name}")
    
    duckdb.sql(f"""
    COPY (
        SELECT *
        FROM read_csv_auto(
            '{csv_path}',
            header = true,
            sample_size = -1
        )
    )
    TO '{parquet_path}'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    );
    """)
    
    print("Quality check - Rows in parquet file:")
    
    duckdb.sql(f"""
    SELECT count(*)
    FROM read_parquet('{parquet_path}')
    """).show()



print("Done creating parquet files from csv.")

Skipping Census_Tracts.csv: bronze_Census_Tracts.parquet already exists
Quality check - Rows in parquet file:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          878 │
└──────────────┘

Skipping Individual_Landmarks.csv: bronze_Individual_Landmarks.parquet already exists
Quality check - Rows in parquet file:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          407 │
└──────────────┘

Skipping taxi.csv: bronze_taxi.parquet already exists
Quality check - Rows in parquet file:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     15406960 │
└──────────────┘

Skipping taxi_sample.csv: bronze_taxi_sample.parquet already exists
Quality check - Rows in parquet file:
┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         5000 │
└──────────────┘

Skipping weatherdata.csv: bronze_weatherdata.parquet already exists
Quality check - Rows in parquet file:
┌──────────────┐
│ count_star() │
│    int64     │